<a href="https://colab.research.google.com/github/SBZ-EDU/Arman2/blob/master/Exhibition_connector_rag1_ipynb_txt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🔧 Exhibition Connector RAG1 — Fixed & Colab-Ready

این نسخه **اصلاح‌شده** کد RAG است که روی Google Colab اجرا می‌شود.

**باگ‌های رفع شده:**
- مسیرهای import قدیمی langchain
- پارامتر `allow_dangerous_deserialization` در FAISS
- اسکیپ دوگانه `\\n` در پرامپت
- عدم مدیریت خطا
- مسیر نادرست فایل اکسل
- `get_relevant_documents` منسوخ شده → `invoke`

## 📦 مرحله ۱: نصب کتابخانه‌ها

In [ ]:
!pip install -q gradio langchain langchain-community langchain-text-splitters \
    faiss-cpu beautifulsoup4 openpyxl transformers sentence-transformers \
    accelerate torch aiohttp

## 🔑 مرحله ۲: تنظیم توکن HuggingFace

برای دسترسی به مدل Llama 3.1، توکن HuggingFace خود را وارد کنید.

**روش توصیه شده:** روی آیکون 🔑 در نوار کناری Colab کلیک کنید و `HF_TOKEN` را اضافه کنید.

In [ ]:
import os

# روش ۱: از Colab Secrets (توصیه می‌شود)
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
    print("✅ توکن از Colab Secrets خوانده شد")
except:
    # روش ۲: ورود دستی
    from getpass import getpass
    HF_TOKEN = getpass("🔑 توکن HuggingFace خود را وارد کنید: ")
    print("✅ توکن وارد شد")

os.environ["HF_TOKEN"] = HF_TOKEN
assert HF_TOKEN, "❌ توکن HuggingFace الزامی است!"
print("✅ توکن تنظیم شد")

## 📥 مرحله ۳: دانلود فایل اکسل

In [ ]:
import requests
from pathlib import Path

DATA_DIR = Path("/content/data")
DATA_DIR.mkdir(exist_ok=True)

xls_path = DATA_DIR / "iran-oil_iran-oil_iran_oil.xlsx"
xls_url = "https://huggingface.co/spaces/sosa123454321/Exhibition-connector-rag1/resolve/main/iran-oil_iran-oil_iran%20oil.xlsx"

if not xls_path.exists():
    print("⏳ در حال دانلود فایل اکسل...")
    r = requests.get(xls_url, timeout=60)
    r.raise_for_status()
    xls_path.write_bytes(r.content)
    print(f"✅ فایل اکسل دانلود شد ({len(r.content):,} بایت)")
else:
    print(f"✅ فایل اکسل از قبل موجود است")

## 📄 مرحله ۴: بارگذاری اسناد

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import WebBaseLoader
from langchain_core.documents import Document
import openpyxl

print("⏳ در حال بارگذاری اسناد...")

# --- بارگذاری ویکی‌پدیا ---
wiki_url = 'https://fa.wikipedia.org/wiki/اوهیا'
wiki_loader = WebBaseLoader(wiki_url)
wiki_docs = wiki_loader.load()
print(f"  ✅ ویکی‌پدیا: {len(wiki_docs)} سند")

# --- بارگذاری اکسل (با openpyxl مستقیم) ---
wb = openpyxl.load_workbook(str(xls_path))
excel_docs = []
for sheet_name in wb.sheetnames:
    ws = wb[sheet_name]
    rows = []
    for row in ws.iter_rows(values_only=True):
        rows.append("\t".join(str(c) if c is not None else "" for c in row))
    content = "\n".join(rows)
    excel_docs.append(Document(
        page_content=content,
        metadata={"source": str(xls_path), "sheet": sheet_name}
    ))
    print(f"  ✅ اکسل: شیت '{sheet_name}' — {ws.max_row} ردیف × {ws.max_column} ستون")

all_docs = wiki_docs + excel_docs
print(f"\n📄 مجموع اسناد: {len(all_docs)}")

# --- تقسیم به چانک‌ها ---
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
splits = text_splitter.split_documents(all_docs)
print(f"✂️ تعداد چانک‌ها: {len(splits)}")

## 🧠 مرحله ۵: ساخت ایندکس FAISS

In [ ]:
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"⏳ در حال بارگذاری مدل Embedding روی {device}...")

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": device},
    encode_kwargs={"normalize_embeddings": True},
)
print(f"✅ مدل Embedding بارگذاری شد")

faiss_index_path = DATA_DIR / "faiss_index"

if faiss_index_path.exists():
    print("⏳ در حال بارگذاری ایندکس FAISS از حافظه...")
    vectorstore = FAISS.load_local(
        str(faiss_index_path), embeddings, allow_dangerous_deserialization=True
    )
    print(f"✅ ایندکس FAISS بارگذاری شد ({vectorstore.index.ntotal} بردار)")
else:
    print("⏳ در حال ساخت ایندکس FAISS...")
    vectorstore = FAISS.from_documents(splits, embeddings)
    vectorstore.save_local(str(faiss_index_path))
    print(f"✅ ایندکس FAISS ساخته و ذخیره شد ({vectorstore.index.ntotal} بردار)")

retriever = vectorstore.as_retriever()

# --- تست بازیابی ---
print("\n🔍 تست بازیابی:")
for q in ["نمایشگاه نفت", "شرکت‌های پتروشیمی"]:
    docs = retriever.invoke(q)
    print(f"  جستجو: \"{q}\" → {len(docs)} نتیجه")
    for i, d in enumerate(docs[:2]):
        print(f"    [{i+1}] {d.page_content[:100].replace(chr(10), ' ')}...")

## 🤖 مرحله ۶: بارگذاری مدل زبانی (LLM)

مدل بر اساس GPU موجود انتخاب می‌شود:
- **GPU ≥ 16GB:** Llama 3.1 8B (بهترین کیفیت)
- **GPU ≥ 8GB:** TinyLlama 1.1B
- **بدون GPU:** GPT-2 (فقط برای تست)

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
import torch

# انتخاب مدل بر اساس GPU
if torch.cuda.is_available():
    gpu_mem = torch.cuda.get_device_properties(0).total_mem / (1024**3)
    print(f"🖥️ GPU: {torch.cuda.get_device_name(0)} ({gpu_mem:.1f} GB)")

    if gpu_mem >= 16:
        LLM_MODEL = "meta-llama/Meta-Llama-3.1-8B-Instruct"
        USE_TOKEN = True
    elif gpu_mem >= 8:
        LLM_MODEL = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
        USE_TOKEN = False
    else:
        LLM_MODEL = "gpt2"
        USE_TOKEN = False
else:
    LLM_MODEL = "gpt2"
    USE_TOKEN = False
    print("⚠️ GPU در دسترس نیست! از مدل کوچک GPT-2 استفاده می‌شود")

print(f"📦 مدل: {LLM_MODEL}")

print(f"⏳ در حال بارگذاری مدل...")
tokenizer = AutoTokenizer.from_pretrained(
    LLM_MODEL, token=HF_TOKEN if USE_TOKEN else None
)
llm_model = AutoModelForCausalLM.from_pretrained(
    LLM_MODEL,
    token=HF_TOKEN if USE_TOKEN else None,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto" if torch.cuda.is_available() else None,
)
llm_pipe = pipeline(
    "text-generation",
    model=llm_model,
    tokenizer=tokenizer,
    max_new_tokens=512,
    temperature=0.8,
    repetition_penalty=1.1,
)
print(f"✅ مدل با موفقیت بارگذاری شد!")

## 🔗 مرحله ۷: تعریف پایپ‌لاین RAG

In [ ]:
def llama_llm(question, context):
    """تولید پاسخ با استفاده از مدل زبانی و متن بازیابی شده"""
    prompt = (
        f"شما یک دستیار هوشمند هستید که به زبان فارسی پاسخ می‌دهید.\n"
        f"سوال: {question}\n\n"
        f"متن مرتبط:\n{context}\n\n"
        f"لطفاً پاسخ دقیق و کامل به زبان فارسی بدهید."
    )

    # محدود کردن طول پرامپت برای جلوگیری از خطای overflow
    max_input_chars = 3000
    if len(prompt) > max_input_chars:
        prompt = prompt[:max_input_chars] + "\n\nلطفاً پاسخ دهید:"

    output = llm_pipe(prompt)[0]['generated_text']
    return output[len(prompt):].strip()


def rag_chain(question):
    """زنجیره RAG: بازیابی + تولید"""
    docs = retriever.invoke(question)
    context = "\n\n".join(doc.page_content for doc in docs)
    return llama_llm(question, context), docs


def get_important_facts(question):
    """تابع اصلی پاسخ‌دهی"""
    if not question.strip():
        return "لطفاً یک سوال معتبر وارد کنید."
    try:
        answer, docs = rag_chain(question)
        return answer
    except Exception as e:
        return f"خطایی رخ داد: {str(e)}"

print("✅ پایپ‌لاین RAG آماده است!")

## 🧪 مرحله ۸: تست پایپ‌لاین RAG

In [ ]:
test_questions = [
    "اطلاعات نفت ایران چیست؟",
    "شرکت‌های پتروشیمی کدامند؟",
    "نمایشگاه نفت تهران چه شرکت‌هایی دارد؟",
]

print("=" * 60)
print("🧪 تست پایپ‌لاین RAG")
print("=" * 60)

for q in test_questions:
    print(f"\n{'─'*50}")
    print(f"❓ سوال: {q}")

    try:
        docs = retriever.invoke(q)
        print(f"📄 تعداد اسناد بازیابی شده: {len(docs)}")

        if docs:
            best = docs[0].page_content[:200].replace("\n", " ")
            print(f"🏆 بهترین نتیجه: {best}...")

        answer, _ = rag_chain(q)
        print(f"🤖 پاسخ: {answer[:500]}")

    except Exception as e:
        print(f"❌ خطا: {e}")

print(f"\n{'='*60}")
print("✅ تست تمام شد!")
print("=" * 60)

## 🖥️ مرحله ۹: رابط کاربری Gradio

رابط کاربری وب اجرا می‌شود. لینک عمومی (`*.gradio.live`) ساخته می‌شود که می‌توانید به اشتراک بگذارید.

In [ ]:
import gradio as gr

WELCOME_MESSAGE = (
    "سلام! من یک هوش مصنوعی هستم که برای کمک به شما در یافتن اطلاعات شرکت‌ها و "
    "نمایشگاه‌ها طراحی شده‌ام. هر سوالی درباره شرکت‌ها یا اطلاعات نمایشگاه دارید، بپرسید."
)

iface = gr.Interface(
    fn=get_important_facts,
    inputs=gr.Textbox(lines=2, placeholder="سوال خود را اینجا وارد کنید..."),
    outputs="text",
    title="هوش مصنوعی پاسخگو به سوالات نمایشگاه",
    description=WELCOME_MESSAGE,
    theme="default",
)

# share=True برای Colab تا لینک عمومی بسازد
iface.launch(share=True, server_name="0.0.0.0", server_port=7860)